In [38]:
import os
import pandas as pd
import base64
import re

In [39]:
#utilities
def df(conn,sql):
    connCur = conn.cursor()
    connCur.execute(sql)
    df = pd.DataFrame(connCur.fetch_pandas_all())
    connCur.close()
    return df

def listJson(conn,sql):
    connCur = conn.cursor()
    connCur.execute(sql)
    df = connCur.fetchall()
    connCur.close()
    return df

def getDDL(conn,sql):
    connCur = conn.cursor()
    result = connCur.execute(sql).fetchone()
    connCur.close()
    return result

def getFileList(path):
    # get file list for path
    filesList = []
    for x in os.listdir(path):
        z = x.upper()
        if z.endswith('.CSV'):
            filesList.append(path + "\\" + x)
    return filesList

def getFile(path):
    f = open(path,'r')
    filein = f.readlines()
    f.close()
    return filein

def writeFile(path,fileOut):
    f = open(path,'w',encoding="utf-8")
    f.write(fileOut)
    f.close()

def writeConfigParser(path,parserFile):
    f = open(path, 'w') 
    parserFile.write(f)


#**************************** LZ to ODS dbt model Utility**********************
# generate helper functions for DBT model

def getJinjaConfig(strategy,cdsView,unique_keys):

	jinjaConfig = """
	{{ config(
	  materialized='incremental',
	  incremental_strategy='"""+ strategy +"""',
	  alias='""" + cdsView + """',
	  unique_key = ['"""+ unique_keys +"""'],
	  transient=false
	) }} """

	return jinjaConfig

#CREATE DBT MODEL
#(inputs are column list for source_cte,transform_cte and tablename )
def getDbtCode(col_s_string,col_t_string,col_type_data_str,tablename,odsSchema,jconfig,ifMerge,join_str):

	DBT_CODE = """


	"""+ jconfig +"""

	with source_data as (
	SELECT
      	
		""" +col_s_string+ """
                	
	from """ + "{{ source('"+ odsSchema + "','" + tablename + "') }} AS " + tablename + " "  """
    """+ join_str +""" 
	"""+ ifMerge +"""

	),
	
	transform_data as (
	SELECT 
		""" +col_t_string+ """,
		'{{ invocation_id }}' AS AUDIT_LOG_ID,
        SHA2('hash_me') as _ROW_HASH
	from source_data
	),
	
	type_data as (
	SELECT
		"""+col_type_data_str+""",
        cast(AUDIT_LOG_ID as varchar(128)) as AUDIT_LOG_ID,
        cast(_ROW_HASH as varchar(128)) as _ROW_HASH 
    from transform_data
	)

	select * from type_data

			"""
	#print(DBT_CODE)
	return DBT_CODE

def createSchema(df_tables_test,lzSchema,odsSchema,schemaPath):
        df_tables_test['schema'] = """\n      - name: """ + df_tables_test['TABLE_NAME'] + """\n        identifier: """ + df_tables_test['TABLE_NAME'] + """\n        description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str = """version: 2\nsources:\n  - name: """ + odsSchema + """\n    description: Landing Zone - """ + lzSchema + """ Schema\n    database: "{{ env_var('SNOWFLAKE_LZ') }}"\n    schema: """+ lzSchema + """\n    tables:"""

        for idx, r in df_tables_test.iterrows():
            schema_str += r['schema'] + """
        """   

        df_tables_test['models'] = """\n  - name: """ + df_tables_test['TABLE_NAME'] + """\n    identifier: """ + df_tables_test['TABLE_NAME'] + """\n    description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str_t ="""\nmodels:"""
        for idx, r in df_tables_test.iterrows():
            schema_str_t += r['models'] + """
        """
        final_schema = schema_str + schema_str_t
        	
        path_create = schemaPath
        filename = "schema.yml"
        path_write = path_create +"\\"+ filename

        # createFile(path_create,filename|)
        writeFile(path_write,final_schema)
        

        df_tables_test['models'] = """\n  - name: """ + df_tables_test['TABLE_NAME'] + """\n    identifier: """ + df_tables_test['TABLE_NAME'] + """\n    description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str_t ="""\nmodels:"""
        for idx, r in df_tables_test.iterrows():
            schema_str_t += r['models'] + """
        """
        final_schema = schema_str + schema_str_t
        	
        path_create = schemaPath
        filename = "schema.yml"
        path_write = path_create +"\\"+ filename

        # createFile(path_create,filename|)
        writeFile(path_write,final_schema)
        

In [40]:
typeDict = {'TEXT':'VARCHAR(16777216)',
            'NUMBER':'NUMBER(38,0)',
            'TIMESTAMP_NTZ':'TIMESTAMP_NTZ(9)',
            'TIMESTAMP_LTZ':'TIMESTAMP_LTZ(9)',
            'BINARY':'VARCHAR(16777216)'
            }
    

df_sf = pd.read_csv('C:\\Users\\222328\\Downloads\\SF Baan table details.csv')
#df_sf.head()
table_list = df_sf['TABLE_NAME'].unique().tolist()

print(len(table_list))

58


In [67]:
#create schema.yml file
df_test = pd.DataFrame(table_list, columns=['TABLE_NAME'])
print(df_test)
createSchema(df_test,"BAAN3_ORACLE_LIVE","COMMON_BAAN_TEE","C:\\Sources\\Repos\\EnterpriseCloud\\dbt\\common_to_pub\\models\\baan_tee\\")

      TABLE_NAME
0   TTCCOM010200
1   TTCCOM020200
2   TTCMCS008200
3   TTCMCS030200
4   TTCMCS032200
5   TTDINV001200
6   TTDPUR040200
7   TTDPUR041200
8   TTDPUR045200
9   TTDRPL100200
10  TTDSLS040200
11  TTDSLS041200
12  TTDSLS045200
13  TTFACP200200
14  TTFACR200200
15  TTFACR200201
16  TTFACR200202
17  TTFACR200203
18  TTFACR200204
19  TTFACR200205
20  TTFACR200206
21  TTFACR200207
22  TTFACR200208
23  TTFACR200209
24  TTFACR400200
25  TTFACR400201
26  TTFACR400202
27  TTFACR400203
28  TTFACR400204
29  TTFACR400205
30  TTFACR400207
31  TTFACR400208
32  TTFACR400209
33  TTFACR401200
34  TTFACR401201
35  TTFACR401202
36  TTFACR401203
37  TTFACR401204
38  TTFACR401205
39  TTFACR401207
40  TTFACR401208
41  TTFACR401209
42  TTFGLD106200
43  TTFGLD106201
44  TTFGLD106202
45  TTFGLD106203
46  TTFGLD106204
47  TTFGLD106205
48  TTFGLD106206
49  TTFGLD106207
50  TTFGLD106208
51  TTFGLD106209
52  TTFGLD410200
53  TTFGLD410208
54  TTFGLD410209
55  TTIITM001200
56  TTIPCS020200
57  TTSSMA1022

In [ ]:
# Build dbt models from TABLE_NAME templates

# read csv template file and build list of required inputs
for tablename in table_list:
    csvData = df_sf.loc[df_sf['TABLE_NAME'] == tablename]
    dbt_strategy = "incremental"
    csvData = csvData.reset_index(drop=True)
    csvData["SCRUBBED_COLUMN_NAMES"]  = csvData["COLUMN_NAME"].str.upper()
    cdsViewName = csvData.at[0,"TABLE_NAME"]
    tableName = csvData.at[0,"TABLE_NAME"] 
    baseTable = csvData.at[0,"TABLE_NAME"]
    csvData = csvData.replace({'DATA_TYPE':typeDict})
    #csvData["columnList"] = csvData["SCRUBBED_COLUMN_NAMES"] + " " + csvData["DATA_TYPE"] + ","
    print(csvData)
    #empty list for transform_cte,source_cte,type_data_cte columns 
    col_t = []
    col_s = []
    col_source = []
    col_type_data = []
    join_str = []
    select_str = []

    for idx,row in csvData.iterrows():
        column_name = row["SCRUBBED_COLUMN_NAMES"]
        if column_name.count('$') == 2:   # to remove the 2nd $ from column name and replace it with _
            # Replace second occurrence of '$' with '_'
            reversed_name = column_name[::-1].replace('$', '_', 1)[::-1]
            col_type_data.append('CAST("{}" AS {}) AS "{}"'.format(reversed_name, row["DATA_TYPE"], reversed_name))
            col_t.append('"{}"'.format(reversed_name))
            col_s.append('{}."{}" AS "{}"'.format(row["TABLE_NAME"], row["COLUMN_NAME"], reversed_name))
        else:
            # No replacement needed, use original column name
            col_type_data.append('CAST("{}" AS {}) AS "{}"'.format(column_name, row["DATA_TYPE"], column_name))
            col_t.append('"{}"'.format(column_name))
            col_s.append('{}."{}" AS "{}"'.format(row["TABLE_NAME"], row["COLUMN_NAME"], column_name))
            
        # col_type_data.append('CAST("{}" AS {}) AS "{}"'.format(row["SCRUBBED_COLUMN_NAMES"], row["DATA_TYPE"], row["SCRUBBED_COLUMN_NAMES"]))
        # col_t.append('"{}"'.format(row["SCRUBBED_COLUMN_NAMES"]))
        # col_s.append('{}."{}" AS "{}"'.format(row["TABLE_NAME"], row["COLUMN_NAME"], row["SCRUBBED_COLUMN_NAMES"]))
 
    col_type_data_str = ', \n \t \t'.join(col_type_data) #typedata cte
    col_t_string = ', \n \t \t'.join(col_t) #transform cte
    col_s_string = ', \n \t \t'.join(col_s)                 #source cte
    join_val = ' \n \t \t'.join(join_str)
    
    # print(join_val)
            
    # build key structure
    dfKeys = csvData.query('PK=="Y"')["COLUMN_NAME"]
    unique_fields=[]
    unique_fields = dfKeys.values.tolist()
    unique_fields_str = "','".join(unique_fields)
    dbt_strategy = dbt_strategy.strip()
    if dbt_strategy == 'incremental':
        dbt_strategy = 'merge'

        unique_fields_qualify = ["get_ignore_case(to_object(JSONFIELD),'{0}')::STRING".format(k) for k in unique_fields]
        unique_fields_qualify_str = ",".join(unique_fields_qualify)

        ifMerge = ""
    else:
        dbt_strategy = 'insert_overwrite'
        ifMerge = ""
        unique_fields_str = ''

    print(join_val)
    #create dbt model jinja config
    jinjaConfig = getJinjaConfig(dbt_strategy,tableName,unique_fields_str)
    dbt_model = getDbtCode(col_s_string,col_t_string,col_type_data_str,tableName,'BAAN_TEE',jinjaConfig,ifMerge,join_val)
    
    #print(dbt_model)
    
    #create dbt file
    dbt_path = "C:\\Sources\\Repos\\EnterpriseCloud\\dbt\\ods_to_common\\models\\baan_tee\\"
    filename = tableName + ".sql"
    path_write = dbt_path + filename

    writeFile(path_write,dbt_model)
    


In [59]:
# create ODS tables from standard TABLE_NAME templates to ODS snow file

# read csv template
overwriteDDLs = True
outstr = "USE DATABASE <<ODS>>;\nUSE SCHEMA " + "BAAN_TEE" + ";\n\n"
#print(filesList)
for tablename in table_list:
    csvData = df_sf.loc[df_sf['TABLE_NAME'] == tablename]
    columnList = ''
    csvData = csvData.reset_index(drop=True)
    tableName = csvData.at[0,"TABLE_NAME"]
    csvData = csvData.replace({'DATA_TYPE':typeDict})
    
    csvData["ScrubbedColumnName"]  = csvData["COLUMN_NAME"].str.upper()
    csvData["ScrubbedColumnName"] = csvData["ScrubbedColumnName"].apply(lambda x: x[::-1].replace('$', '_', 1)[::-1] if x.count('$') == 2 else x)
    csvData["columnList"] = '"' + csvData["ScrubbedColumnName"] + '" ' + csvData["DATA_TYPE"] + ","
    
    for idx,row in csvData.iterrows():
        columnList += row["columnList"] + '\n\t'
    if overwriteDDLs:
        ddlAction = 'create or replace TABLE ' + tableName + ' (\n\t'
    else:
        ddlAction = 'create TABLE ' + tableName  + ' if not exists (\n\t'
    controlColumns = 'AUDIT_LOG_ID varchar(128),\n\t_ROW_HASH varchar(128)\n);\n\n'
    
    outstr +=  ddlAction + columnList + controlColumns
    #print(outstr)
    #break

#print(ddlString) ## too big to print, check in file
snowODS_path = "C:\\Sources\\Repos\\EnterpriseCloud\\Snow\\K.31678.1__baan_tee_tables.sql"
writeFile(snowODS_path,outstr)


In [ ]:
## create pub COMMON_ views
%run ..\\Python\\dbtTools\\dbtGeneratePubViews.py "COMMON" "BAAN_TEE" "C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models" "pritesh.x.gandhi@us.tel.com"